# RFDiffusion3-Notebook: *de novo* Protein design

**AI-powered protein design and structure generation** Interactive Jupyter notebook for de novo protein backbone generation, sequence extraction, and structure export using the RFD3 diffusion model.

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python)
![CUDA](https://img.shields.io/badge/CUDA-Enabled-green?logo=nvidia)
![RFD3](https://img.shields.io/badge/Model-RFD3-purple)
![Platform](https://img.shields.io/badge/Platform-Colab%20-lightgrey?logo=googlecolab)

---

## RFdiffusion version
**RFdiffusion3 (RFD3)**

Installed via rc-foundry (foundry install rfd3)


---

## Overview

RFD3-Notebook is an interactive Google Colab pipeline that integrates the RFdiffusion3 (RFD3) model for all-atom protein design. It enables generation of novel protein backbones under structural constraints, optionally conditioned on ligands, followed by downstream sequence extraction and structural export for analysis.


---

## Workflow

1. **Setup** – Install rc-foundry environment and load RFD3 / MPNN / RF3 dependencies.
2. **Input** - Provide protein scaffold (PDB) and optional ligand definitions.
3. **Design** - Generate novel protein backbones using diffusion-based sampling (RFD3).
4. **Conditioning** - Apply structural constraints (contigs, fixed atoms, ligand binding sites).
5. **Samplin** - Run multiple batches to generate diverse protein designs.
5. **Extraction** - Convert CIF outputs to PDB and extract amino acid sequences.
5. **Export** - Zip and download generated structures and sequences.

## Run Time Setup

**GPU Required:** Before running, enable GPU runtime:
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** (or better)
3. Click **Save**

# Environment Setup

In [ ]:
# install dependencies (skip if already done)
import os

# set environment variables
os.environ['CCD_MIRROR_PATH'] = ''
os.environ['PDB_MIRROR_PATH'] = ''

if not os.path.isfile("FOUNDRY_READY"):
    print("Installing rc-foundry...")

    # uninstall torchvision first to avoid operator conflicts
    os.system("pip uninstall -y torchvision")

    # install rc-foundry
    os.system("pip install -q 'rc-foundry[all]'")

    # mark as ready
    os.system("touch FOUNDRY_READY")

    print("Done!")
else:
    print("rc-foundry already installed.")

Installing rc-foundry...
Done!


In [ ]:
# Download model weights (skips already-downloaded models automatically)
# In total, ~6GB (3GB for RFD3, 3GB for RF3, <100MB for MPNN); may take a few minutes depending on your connection speed
os.system("foundry install rfd3 ligandmpnn rf3")

0

# Model information

All models are unified through [AtomWorks](https://github.com/RosettaCommons/atomworks) (for both inference and training), relying on Biotite `AtomArray` objects.


In [ ]:
import warnings
warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",


# Input Structure Upload

A protein structure file (PDB) is uploaded and used as the starting protein for RFD3 generation. The uploaded structure defines the regions that remain fixed and the regions that will be redesigned during diffusion sampling.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving cnbd_with_camp.pdb to cnbd_with_camp.pdb


## All-Atom Generation with RFD3

RFdiffusion3 (RFD3) generates *de novo* all-atom proteins that meet specific conditioning requirements.

**Parameters Used** *(many more are available for protein design tasks)*:
- `input`: Input PDB structure containing the protein scaffold and optional ligand coordinates.
- `contig`: Defines the structural regions that remain fixed and the regions that will be generated de novo.
- `ligand`: Three-letter residue code identifying the ligand present in the input structure.
- `select_fixed_atoms`: Specifies atoms that remain fixed during diffusion sampling and are used as structural anchors.
- `select_burried`: Defines ligand atoms that should remain buried or contacted by the generated protein region, encouraging preservation of binding interactions.
- `diffusion_batch_size`: Number of structures to generate per batch
- `n_batches`: Number of batches to generate.

**Random design**: Parameters `input` and `contig` are used for the random design.

**Ligand specific design**: Parameters `input`, `contig`, `ligand`, `select_fixed_atoms` and `select_burried` are used for the ligand specific design.

Paramters `diffusion_batch_size` and `n_batches` are used for both Random and Ligand specific designs.

**Outputs:** Dictionary of RFD3Output objects.

In [ ]:
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

# set seed for reproducibility
seed_everything(0)

# configure RFdiffusion3 for protein design
config = RFD3InferenceConfig(
    specification={

        # input structure containing the protein and ligand for ligand specific designs
        # input structure dooes not have to contain the ligand for random designs
        'input': "/content/cnbd_with_camp.pdb",

        # define protein with a 15-residue de novo gap to design
        'contig': "A157-267,15-15,A283-305",

        # ligand residue identifier present in the input structure
        'ligand': "CMP",

        # keep ligand atoms and scaffold backbone anchor atoms fixed
        'select_fixed_atoms': {
            'CMP': "P,O1P,O2P,N1,C2,N3,C4,C5,C6,N6,N7,C8,N9,C1',C2',O2',C3',O3',C4',O4',C5',O5'",
            'A157-267': "CA",
            'A283-305': "CA"
        },

        # encourage the generated region to interact with the ligand
        'select_buried': {
            'CMP': "P,O1P,O2P,N1,C2,N3,C4,C5,C6,N6,N7,C8,N9,C1',C2',O2',C3',O3',C4',O4',C5',O5'"
        }
    },
    # number of structures to generate per batch
    diffusion_batch_size=2,
)

# initialize the inference engine
model = RFD3InferenceEngine(**config)

# run generation and save outputs
outputs = model.run(
    inputs=None,
    out_dir="outputs/epac1_whole_camp",
    n_batches=10,
)

INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO:rfd3.engine:[rank: 0] Outputs will be written to /content/outputs/epac1_whole_camp.
INFO:rfd3.engine:[rank: 0] Found 0 existing example IDs in the output directory (0 total).
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:rfd3.engine:[rank: 0] Finished inference batch in 38.33 seconds.
INFO:rfd3.engine:[rank: 0] Outputs for _4_model_0 written to /content/outputs/epac1_whole_camp/_4_model_0.
INFO:rfd3.engine:[rank: 0] Outputs for _4_model_1 written to /content/outputs/epac1_whole_camp/_4_model_1.
INFO:rfd3.engine:[rank: 0] Finished inference batch in 39.61 seconds.
INFO:rfd3.engine:[rank: 0] Outputs for _1_model_0 written to /content/outputs/epac1_whole_camp/_1_model_0.
INFO:rfd3.engine:[rank: 0] Outputs for _1_model_1 written to /content/outputs/epac1_whole_camp/_1_model_1.
INFO:rfd3.engine:[rank: 0] Finished in

# Structure Conversion and Sequence Extraction
Generated RFdiffusion3 structures are exported as CIF files. This section automatically locates all generated designs, converts them to PDB format, extracts the corresponding amino acid sequences, and stores the results in a structured dictionary. All generated structures are then collected into a ZIP file.

In [ ]:
import os
import glob
import shutil
import subprocess
from biotite.structure.io.pdb import PDBFile
from biotite.structure.io.pdbx import CIFFile, get_structure
from biotite.structure import get_residues
from google.colab import files

# directory containing RFdiffusion3 outputs
output_dir = "/content/outputs/epac1_whole_camp"

# directory used to store converted PDB structures
generated_dir = "generated_structures"
os.makedirs(generated_dir, exist_ok=True)

# locate all compressed CIF structure files
cif_gz_files = sorted(glob.glob(f"{output_dir}/**/*.cif.gz", recursive=True))

# amino-acid conversion table
# converts three-letter residue names to one-letter sequence codes
aa_map = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C",
    "GLN":"Q","GLU":"E","GLY":"G","HIS":"H","ILE":"I",
    "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"
}

# store extracted sequences
all_sequences = {}

# structure counter for unique filenames
counter = 1

# process all generated structures
for cif_gz_path in cif_gz_files:

    cif_gz_name = os.path.basename(cif_gz_path)

    # decompress CIF file
    temp_cif = cif_gz_name.replace(".gz", "")

    subprocess.run(["gunzip", "-c", cif_gz_path],
                    stdout=open(temp_cif, "wb"), check=True)

     # load structure from CIF file
    cif_obj = CIFFile.read(temp_cif)
    atom_array = get_structure(cif_obj, model=1)

    # export structure as PDB
    pdb_name = f"{generated_dir}/structure_{counter}.pdb"
    pdb_obj = PDBFile()
    pdb_obj.set_structure(atom_array)
    pdb_obj.write(pdb_name)

    # extract amino-acid sequence
    _, residue_names = get_residues(atom_array)
    sequence = "".join(aa_map.get(res, "") for res in residue_names)

    # store sequence in dictionary
    all_sequences[f"structure_{counter}"] = sequence

    counter += 1

    # remove temporary decompressed CIF file
    if os.path.exists(temp_cif):
        os.remove(temp_cif)

# display extracted sequences
print(all_sequences)

# create ZIP archive containing all generated PDB structures
shutil.make_archive("generated_structures", "zip", generated_dir)

# download archive
files.download("generated_structures.zip")

Found 20 .cif.gz files

Saved structure 1 (149 AA) ← _0_model_0.cif.gz
Saved structure 2 (149 AA) ← _0_model_1.cif.gz
Saved structure 3 (149 AA) ← _1_model_0.cif.gz
Saved structure 4 (149 AA) ← _1_model_1.cif.gz
Saved structure 5 (149 AA) ← _2_model_0.cif.gz
Saved structure 6 (149 AA) ← _2_model_1.cif.gz
Saved structure 7 (149 AA) ← _3_model_0.cif.gz
Saved structure 8 (149 AA) ← _3_model_1.cif.gz
Saved structure 9 (149 AA) ← _4_model_0.cif.gz
Saved structure 10 (149 AA) ← _4_model_1.cif.gz
Saved structure 11 (149 AA) ← _5_model_0.cif.gz
Saved structure 12 (149 AA) ← _5_model_1.cif.gz
Saved structure 13 (149 AA) ← _6_model_0.cif.gz
Saved structure 14 (149 AA) ← _6_model_1.cif.gz
Saved structure 15 (149 AA) ← _7_model_0.cif.gz
Saved structure 16 (149 AA) ← _7_model_1.cif.gz
Saved structure 17 (149 AA) ← _8_model_0.cif.gz
Saved structure 18 (149 AA) ← _8_model_1.cif.gz
Saved structure 19 (149 AA) ← _9_model_0.cif.gz
Saved structure 20 (149 AA) ← _9_model_1.cif.gz
{'structure_1': 'EELAEAVA

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>